# Tutorial -- all about decorators

1. What are decorators, and how do they work?
2. Creating simple decorators
3. Creating decorators that modify inputs
4. Creating decorators that modify outputs
5. Multiple decorators
6. Decorators that take arguments
7. Improving our decorators with ... decorators!
8. (If we have time) decoratoting classes, as well

# DRY -- don't repeat yourself!

In [1]:
def a():
    return f'a!\n'

def b():
    return f'b!\n'

print(a())    
print(b())

a!

b!



In [2]:
# corporate hq says: every function's output must start and end with ----

lines = '-' * 60 + '\n'

def a():
    return f'{lines}a!\n{lines}'

def b():
    return f'{lines}b!\n{lines}'

print(a())    
print(b())

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------



In [3]:
# Option 2: instead of calling the function directly,
# I'll call it via another function. That second function
# will then insert the lines

lines = '-' * 60 + '\n'

def with_lines(func):
    return f'{lines}{func()}{lines}'

def a():
    return f'a!\n'

def b():
    return f'b!\n'

print(with_lines(a))
print(with_lines(b))  

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------



In [4]:
# option 3: nested function -- a "closure"

lines = '-' * 60 + '\n'

def with_lines(func):
    def wrapper():
        return f'{lines}{func()}{lines}'
    return wrapper

def a():
    return f'a!\n'
with_lines_a  = with_lines(a)

def b():
    return f'b!\n'
with_lines_b = with_lines(b)

print(with_lines_a())
print(with_lines_b()) 

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------



In [5]:
# option 4: instead of assigning to with_lines_a and with_lines_b,
# just assign to a and b!

lines = '-' * 60 + '\n'

def with_lines(func):
    def wrapper():
        return f'{lines}{func()}{lines}'
    return wrapper

def a():
    return f'a!\n'
a  = with_lines(a)

def b():
    return f'b!\n'
b = with_lines(b)

print(a())
print(b())

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------



In [ ]:
# option 5: Use Python decorator syntax

lines = '-' * 60 + '\n'

def with_lines(func):
    def wrapper():
        return f'{lines}{func()}{lines}'
    return wrapper

@with_lines    # this is precisely the same as line 13's action
def a():
    return f'a!\n'
# a  = with_lines(a)

@with_lines   # this is precisely the same as line 18's actions
def b():
    return f'b!\n'
# b = with_lines(b)

print(a())
print(b())

# What is a decorator?

1. It's a function
2. That takes a function as an argument
3. That returns a function
4. That is assigned back to the original function, and runs in its place

# Who cares?

A decorator allows us to hijack a function at two points (a) definition time and (b) runtime. We can do whatever we want at either of those points in time.

1. Replace a function, perhaps if the permissions are wrong
2. Only allow a function to run under certain circumstances
3. Change/filter inputs
4. Change/filter outputs
5. Log information about the function and what it's doing

Everything we do with decorators could be done without them. But decorators give us tight syntax to do these things, and let us think at a higher level of abstraction.

In [11]:

lines = '-' * 60 + '\n'

def with_lines(func):
    def wrapper(*args, **kwargs):
        return f'{lines}{func(*args, **kwargs)}{lines}'
    return wrapper

@with_lines    # this is precisely the same as line 13's action
def a():
    return f'a!\n'
# a  = with_lines(a)

@with_lines   # this is precisely the same as line 18's actions
def b():
    return f'b!\n'
# b = with_lines(b)

@with_lines
def add(x, y):
    return f'{x} + {y} = {x+y}\n'

print(a())
print(b())
print(add(10, 30))

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------

------------------------------------------------------------
10 + 30 = 40
------------------------------------------------------------



# How to write a decorator

1. Write the decorator function (what you will use with @) as a regular function that takes a single argument, which I normally call `func`. This will be the decorated function.
2. Inside of the decorator function, define an inner function, usually called `wrapper`. it should take `*args` and `**kwargs`.
3. `wrapper` can do whatever it wants -- but usually it'll end up calling the original function (available as `func`), and then storing/returning the result
4. The decorator function then needs to `return wrapper` -- we aren't invoking `wrapper`, just returning a reference to it!
5. We can apply `@decorator` to any function we want to change behavior in.

# Exercise: Timing

1. Write a decorator, called `timefunc`, that checks how long it takes for a function to run. The function will run normally, with its usual arguments, and returning its usual values, *but* we will keep track of how long each execution takes.
2. The time will be stored to a file called `timing.txt`.
3. You can get the current Unix time with `time.time()`
4. You can get the name of the function with `__name__`.
5. You can write two slow functions -- I usually write `slow_mul` and `slow_add` that just add someething like `time.sleep(random.randint(0, 3))` at the start of the function.

Practice system: 
- Practice system: https://practice.lernerpython.com/classroom/24d84164d2/
- Exercise 1: https://practice.lernerpython.com/classroom/24d84164d2/ex-1

In [14]:
import time
import random

def timefunc(func):   # outer decorator function always takes a func argument
    def wrapper(*args): # wrapper takes any number of positional arguments
        start_time = time.time()
        result = func(*args)
        total_time = time.time() - start_time

        with open('timing.txt', 'a') as outfile:
            outfile.write(f'{func.__name__}\t{start_time}\t{total_time}\n')

        return result
    return wrapper

@timefunc
def slow_add(x, y):
    time.sleep(random.randint(0, 3))
    return x + y

@timefunc
def slow_mul(x, y):
    time.sleep(random.randint(0, 3))
    return x * y

print(slow_add(2, 3))
print(slow_mul(4, 5))
print(slow_add(6, 7))
print(slow_mul(8, 9))

5
20
13
72


In [15]:
!cat timing.txt

slow_add	1784017290.386604	1.0050768852233887
slow_mul	1784017291.3936179	2.005068063735962
slow_add	1784017293.399713	2.0050601959228516
slow_mul	1784017295.405459	1.0024139881134033


# Caching

What if we set things up such that `wrapper` (the innner function) wil check the arguments of the function that is being run -- and if it has seen these arguments before, then it doesn't run the function. Rather, it returns the results from that previous time, which were cached?

In other words:
- When a function is invoked, we look in a cache for the arguments
- If the arguments are not there, we invoke the function and store the arguments + return value
- No matter what, we can now return the result, based on the arguments

This is known as memoization.

In [17]:
def memoize(func):
    cache = {}

    def wrapper(*args):
        if args not in cache:   # have we seen these arguments before? No? well...
            cache[args] = func(*args)  # call the function, store result to the cache

        return cache[args]    # cache is definitely populated!
    return wrapper

@memoize
def slow_add(x, y):
    time.sleep(random.randint(0, 3))
    return x + y

@memoize
def slow_mul(x, y):
    time.sleep(random.randint(0, 3))
    return x * y

print(slow_add(2, 3))
print(slow_mul(2, 3))
print(slow_add(2, 3))
print(slow_mul(2, 3))

5
6
5
6
